# Drive Cleanup — free up space by removing old NPZs

Run this when the 200 GB Drive quota is full.

**Strategy.** Walk `MyDrive/ARPG-assets/results/`, group NPZs by phase, print totals, then offer a guided deletion. Nothing is deleted unless you explicitly uncomment a `delete_group(...)` call.

**Safe to re-run.** It only reports + offers deletions; it never auto-deletes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import defaultdict

DRIVE_ROOT = Path('/content/drive/MyDrive/ARPG-assets')
RESULTS_ROOT = DRIVE_ROOT / 'results'
print(f'Scanning {RESULTS_ROOT}…')

## 1. Inventory — what's on Drive and how big

In [ ]:
# Build groups: {top-level results subdir: [(size_bytes, path), ...]}
groups = defaultdict(list)
total_bytes = 0
for root_str, _, files in __import__('os').walk(str(RESULTS_ROOT)):
    root = Path(root_str)
    for f in files:
        if not f.endswith('.npz'):
            continue
        full = root / f
        size = full.stat().st_size
        total_bytes += size
        try:
            phase = full.relative_to(RESULTS_ROOT).parts[0]
        except ValueError:
            phase = '(other)'
        groups[phase].append((size, full))

# Print summary
print(f'\nTotal NPZ data on Drive: {total_bytes/1e9:.1f} GB')
print(f'Across {sum(len(v) for v in groups.values())} files in {len(groups)} top-level folders\n')

# Per-group breakdown
for phase, items in sorted(groups.items(), key=lambda kv: -sum(s for s, _ in kv[1])):
    sub_total = sum(s for s, _ in items) / 1e9
    print(f'  {sub_total:>7.1f} GB   {len(items):>3} files   {phase}')

# Also show the biggest individual files for inspection
all_files = sorted([(s, p) for items in groups.values() for s, p in items], reverse=True)
print(f'\nTop 15 largest NPZ files:')
for size, path in all_files[:15]:
    rel = path.relative_to(RESULTS_ROOT)
    print(f'  {size/1e9:>6.2f} GB   {rel}')

## 2. Deletion — uncomment a `delete_group(...)` line to actually free space

Each `delete_group` call is **destructive**. Read the size first, then uncomment what you want gone.

**Recommended keep-list** for the final paper:
- `phase3-fid50k/` — Phase 3 50K NPZs (3 vanilla + 2 RTR seed-0) → keep (small, useful for paper figures)
- `final-paper/random-deferral/` — Phase 5 Item 1 RTR seed-0 → keep
- Everything else → safe to delete; you have the FID values in the CSVs.

In [ ]:
def delete_group(phase_name, dry_run=True):
    """Delete all NPZs under RESULTS_ROOT/<phase_name>/.
    
    dry_run=True prints what would be deleted without deleting.
    dry_run=False actually unlinks the files.
    """
    target = RESULTS_ROOT / phase_name
    if not target.exists():
        print(f'Path does not exist: {target}')
        return
    npzs = [p for p in target.rglob('*.npz')]
    total = sum(p.stat().st_size for p in npzs) / 1e9
    action = 'WOULD DELETE' if dry_run else 'DELETING'
    print(f'{action}: {len(npzs)} NPZs ({total:.1f} GB) under {target}')
    for p in npzs:
        rel = p.relative_to(RESULTS_ROOT)
        if dry_run:
            print(f'   would unlink: {rel}')
        else:
            p.unlink()
    if not dry_run:
        print(f'Freed {total:.1f} GB')

# ============================================================
# Example deletions — review then uncomment ONE line at a time.
# Start with dry_run=True to confirm, then flip to dry_run=False.
# ============================================================

# delete_group('pilot-20260421', dry_run=True)
# delete_group('pilot-20260421/phase3-fid50k', dry_run=True)
# delete_group('final-paper/multi-seed', dry_run=True)
# delete_group('final-paper/cap-sweep', dry_run=True)
# delete_group('final-paper/wallclock', dry_run=True)

# Once you've reviewed the dry_run output, flip to dry_run=False:
# delete_group('final-paper/wallclock', dry_run=False)

print('Read the cell above and uncomment the deletions you want.')

## 3. Re-check after deletions

In [ ]:
# Re-scan to confirm space freed
import os
total_bytes = 0
n_files = 0
for root_str, _, files in os.walk(str(RESULTS_ROOT)):
    for f in files:
        if f.endswith('.npz'):
            total_bytes += (Path(root_str) / f).stat().st_size
            n_files += 1
print(f'After cleanup: {n_files} NPZs, {total_bytes/1e9:.1f} GB on Drive')